# v_r_v1 Evaluation

Dieses Notebook prueft das gespeicherte Modell `models/v_r_v1_eval/output/v_r_v1.pkl` gegen die vorhandenen `Vorhand`/`Rueckhand`-Eventlabels und zeigt die wichtigsten Kennzahlen fuer den Vergleich mit anderen Modellen.

## Hinweis

Die aktuelle Eventmenge scheint identisch mit den im Modell gespeicherten `training_samples` zu sein. Eine `1.0`-Bewertung ist daher sehr wahrscheinlich **In-Sample** und nicht automatisch ein fairer Generalisierungswert.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "vr_model_evaluation").exists():
            return candidate
    raise FileNotFoundError("Repo root mit src/vr_model_evaluation wurde nicht gefunden.")

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.vr_model_evaluation.config import VRModelEvaluationConfig
from src.vr_model_evaluation.runner import evaluate_frozen_model, load_model_payload, save_confusion_matrix_plot

config = VRModelEvaluationConfig.from_json(repo_root / "models/v_r_v1_eval/config.json")
payload = load_model_payload(config)
config

In [ ]:
metadata = {
    key: value
    for key, value in payload.items()
    if key != "model"
}
display(pd.DataFrame([metadata]))

In [ ]:
result = evaluate_frozen_model(config)
display(pd.DataFrame([result.overall_metrics]))
display(result.session_metrics)
display(result.label_summary)

In [ ]:
display(result.classification_report)
display(result.predictions.head(20))

In [ ]:
display(result.feature_importances.head(20))

In [ ]:
figure = save_confusion_matrix_plot(result.confusion_matrix)
figure